# Isolation Forest 

Isolation Forest is an unsupervised learning model. It does not use labels during the training. It learns what normal behaiour looks like and finds the one that looks different. 
Here i will try to show how well Isolation Forest can
separate fraud transactions and normal transactions. 

In [2]:
import pandas as pd

df = pd.read_csv("../data/transactions.csv")
df.head()

,transaction_id,customer_id,amount,hour,day_of_week,distance_from_home,merchant_category,is_fraud
0,TXN_00000000,861,321.657107,12,6,8.349037,restaurant,0
1,TXN_00000001,3773,1944.835714,18,2,4.499823,grocery,0
2,TXN_00000002,3093,14551.485133,11,2,18.237703,grocery,0
3,TXN_00000003,467,977.794274,6,1,2.069479,restaurant,0
4,TXN_00000004,4427,106.712706,20,1,13.467285,restaurant,0


## Feature Selection

Only numerical and behaviour related features are used here. This helps me in understanding how the model works using these features alone.

In [3]:
features = [
    "amount",
    "hour",
    "day_of_week",
    "distance_from_home"
]

X = df[features]

In [4]:
from sklearn.ensemble import IsolationForest

iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.01,
    random_state=42
)

iso_forest.fit(X)


IsolationForest(contamination=0.01, random_state=42)

In [5]:
df["anomaly_score"] = iso_forest.decision_function(X)
df["anomaly_label"] = iso_forest.predict(X)

In [6]:
df[["is_fraud", "anomaly_label"]].value_counts()

is_fraud  anomaly_label
0          1               97855
1          1                1145
0         -1                 888
1         -1                 112
Name: count, dtype: int64

Unlike One-Class SVM, Isolation Forest does not measure distances between data points.Isolation Forest is not sensitive to feature magnitudes so feature scaling is not required

When anomaly_label = 1 then it is predicted normal 
When anomaly_label = -1 then it is predicted anomaly 

correctly predicted normal transactions are : 97855---> actual users not disturbed

fraud transactions but predicted as normal transactions : 1145---> false -ve

normal transactions but predicted as fraudulent transactions : 888--->false +ve

correctly predicted fraudulent transactions : 112---> correctly detected fraud 

In [7]:
df["predicted_fraud"] = (df["anomaly_label"] == -1).astype(int)
df[["is_fraud", "predicted_fraud"]].head()

,is_fraud,predicted_fraud
0,0,0
1,0,0
2,0,1
3,0,0
4,0,0


### Conversion from 1/-1 to 0/1 :
the conversion is needed because evaluation metrics work on 0s and 1s, but Isolation Forest outputs 1 means normal and -1 means anomaly.

In [8]:
from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(df["is_fraud"], df["predicted_fraud"])
recall = recall_score(df["is_fraud"], df["predicted_fraud"])
f1 = f1_score(df["is_fraud"], df["predicted_fraud"])

precision, recall, f1

(0.112, 0.08910103420843278, 0.09924678777137794)

### Precision, Recall, F1

Precision says how many transactions flagged as fraud were actually
fraudulent. In here, low precision means more innocent people get blocked (false +ve)
Recall says how many actual fraud cases were detected.
F1-score balances precision and recall.

In [9]:
df["anomaly_score"]

0        0.145484
1        0.199072
2       -0.017031
3        0.194295
4        0.130520
           ...   
99995    0.188659
99996    0.147445
99997    0.150005
99998    0.137572
99999    0.188265
Name: anomaly_score, Length: 100000, dtype: float64

In [10]:
from sklearn.metrics import roc_auc_score

# Invert anomaly score so higher = more likely fraud
auc = roc_auc_score(df["is_fraud"], -df["anomaly_score"])

auc

0.7465838912553229

### ROC–AUC
ROC-AUC tells how good the model is at separating the frauds from normal users. If AUC is higher then the model is better at understanding the difference. 

### Isolation Forest – Metric Analysis

The Isolation Forest baseline shows low precision and recall,
which indicates that many fraud cases are missed and many normal transactions are flagged as suspicious.

But the ROC–AUC score says that the model is able to rank
fraudulent transactions higher than normal ones well.

In [11]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(iso_forest, "../models/isolation_forest.pkl")

['../models/isolation_forest.pkl']